In [ ]:
import torch
from functools import partial
from typing import List

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

# Kernel Tutorial: Building an Efficient Elementwise Add Kernel with CuTe DSL

This tutorial demonstrates how to implement and optimize a GPU elementwise addition kernel using the CuTe DSL. 

## Learning Objectives

In this tutorial, you will learn building an efficient elementwise kernel in CuTe DSL step by step:
- How to implement basic GPU kernels using CuTe DSL in basic CUDA techniques
- How to benchmark performance of the kernel
- How to tile and partition tensor and map to basic CuTe Layout
- What it Thread & Value Layout and mapping from thread & value index to logical coordinate
- How to implement advanced kernel with TV layout and tune performance to achieve peak performance

## Understanding Elementwise Addition

Elementwise addition is a fundamental operation in linear algebra and deep learning. Given two tensors of the same shape, the operation performs element-wise addition to produce a result tensor of the same shape.

For two 2D tensors $A$ and $B$ of shape $(M, N)$, the elementwise addition operation $C = A + B$ is defined as:

$
   C_{i,j} = A_{i,j} + B_{i,j}
$

where:
- $i \in [0, M-1]$ represents the row index
- $j \in [0, N-1]$ represents the column index
- $A_{i,j}$, $B_{i,j}$, and $C_{i,j}$ are the elements at position $(i,j)$ in tensors $A$, $B$, and $C$ respectively

This operation has several important characteristics:
1. **Parallelizable**: Each element can be computed independently
2. **Memory-bound**: Performance limited by memory bandwidth rather than compute
3. **Coalescing-sensitive**: Efficiency depends on memory access patterns
4. **Vectorization-friendly**: Multiple elements can be processed together

## Naive Elementwise Add Kernel

Let's start with a naive implementation to establish a baseline before exploring optimizations.

In [ ]:
# 内核教程：使用CuTe DSL构建高效的元素级加法内核

本教程演示如何使用CuTe DSL实现和优化GPU元素级加法内核。

## 学习目标

在本教程中，您将逐步学习使用CuTe DSL构建高效的元素级内核：
- 如何使用CuTe DSL实现基本的GPU内核，使用基本的CUDA技术
- 如何基准测试内核的性能
- 如何平铺和分区张量，并映射到基本的CuTe布局
- 什么是线程和值布局，以及从线程和值索引到逻辑坐标的映射
- 如何使用TV布局实现高级内核，并调整性能以达到峰值性能

## 理解元素级加法

元素级加法是线性代数和深度学习中的基本操作。给定两个相同形状的张量，该操作执行元素级加法以产生相同形状的结果张量。

对于两个形状为$(M, N)$的2D张量$A$和$B$，元素级加法操作$C = A + B$定义为：

$
    C_{i,j} = A_{i,j} + B_{i,j}
$

其中：
- $i \in [0, M-1]$ 表示行索引
- $j \in [0, N-1]$ 表示列索引
- $A_{i,j}$、$B_{i,j}$ 和 $C_{i,j}$ 分别是张量$A$、$B$ 和 $C$ 中位置$(i,j)$处的元素

此操作具有几个重要特征：
1. **可并行化**：每个元素可以独立计算
2. **内存受限**：性能受内存带宽限制而不是计算
3. **合并敏感**：效率取决于内存访问模式
4. **向量化友好**：多个元素可以一起处理

## 朴素的元素级加法内核

让我们从一个朴素的实现开始，以在探索优化之前建立一个基准。

In [ ]:
# Basic Kernel Implementation
# ---------------------
# This is our first implementation of the elementwise add kernel.
# It follows a simple 1:1 mapping between threads and tensor elements.


@cute.kernel
def naive_elementwise_add_kernel(
    gA: cute.Tensor,  # Input tensor A
    gB: cute.Tensor,  # Input tensor B
    gC: cute.Tensor,  # Output tensor C = A + B
):
    # Step 1: Get thread indices
    # ------------------------
    # CUDA threads are organized in a 3D grid of thread blocks
    # Here we only use the x-dimension for simplicity
    tidx, _, _ = cute.arch.thread_idx()  # Thread index within block (0 to bdim-1)
    bidx, _, _ = cute.arch.block_idx()  # Block index in grid (0 to grid_dim-1)
    bdim, _, _ = cute.arch.block_dim()  # Number of threads per block

    # Calculate global thread index
    # This gives each thread a unique ID across all blocks
    thread_idx = bidx * bdim + tidx  # Global thread ID

    # Step 2: Map thread index to tensor coordinates
    # -------------------------------------------
    # Each thread will process one element of the input tensors
    m, n = gA.shape  # Get tensor dimensions (M rows × N columns)

    # Convert linear thread index to 2D coordinates:
    # - ni: column index (0 to n-1)
    # - mi: row index (0 to m-1)
    ni = thread_idx % n  # Column index (faster varying dimension)
    mi = thread_idx // n  # Row index (slower varying dimension)

    # Step 3: Load and process data
    # ---------------------------
    # Load values from input tensors
    # The tensor layout automatically handles the conversion from
    # logical indices (mi, ni) to physical memory addresses
    a_val = gA[mi, ni]  # Load element from tensor A
    b_val = gB[mi, ni]  # Load element from tensor B

    # Step 4: Store result
    # ------------------
    # Write the sum back to the output tensor
    gC[mi, ni] = a_val + b_val

### Structure of the Kernel

The naive kernel implementation follows a straightforward but effective structure for parallel processing on the GPU. Here's a detailed breakdown of how it works:

1. **Thread Organization and Indexing**
   - Each CUDA thread is uniquely identified using a combination of:
     * `thread_idx` (tidx): Thread index within a block (0 to bdim-1)
     * `block_idx` (bidx): Block index in the grid
     * `block_dim` (bdim): Number of threads per block
   - Global thread ID is calculated as: `thread_idx = bidx * bdim + tidx`

2. **Coordinate Mapping**
   - The kernel maps each thread's global ID to 2D tensor coordinates:
     * `ni = thread_idx % n` (column index - faster varying)
     * `mi = thread_idx // n` (row index - slower varying)
   - This mapping ensures coalesced memory access by having adjacent threads access adjacent memory locations

3. **Memory Access Pattern**
   - Each thread:
     * Loads one element from tensor A: `a_val = gA[mi, ni]`
     * Loads one element from tensor B: `b_val = gB[mi, ni]`
     * Performs addition: `a_val + b_val`
     * Stores result to tensor C: `gC[mi, ni] = result`
   - Memory Considerations
     * Uses 1:1 thread-to-element mapping
     * Memory accesses are coalesced when threads in a warp access consecutive elements
     * No explicit use of shared memory or register blocking
     * Limited ability to hide memory latency due to single element processing

This naive implementation provides a baseline for understanding more optimized versions that follow, which introduce:
- Vectorized memory access
- Thread and value (TV) layouts
- Advanced tiling strategies
- Custom binary operations

For more details about coalesced memory access, please read: https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/#coalesced-access-to-global-memory


### Kernel Launch Configuration and Testing

This section demonstrates how to:
1. Configure and launch the kernel with `cute.jit` function
2. Set up test data with `torch`
3. Verify correctness

**Launch Configuration**
  - Uses 256 threads per block (common choice for good occupancy)
  - Grid size calculated based on total elements: `(m * n) // threads_per_block`
  - Single dimension block and grid configuration for simplicity

#### Host JIT function to launch kernel

In [ ]:

### 内核结构

朴素内核实现遵循了一个简单但有效的GPU并行处理结构。下面是它如何工作的详细分解：

1. **线程组织和索引**
    - 每个CUDA线程使用以下组合唯一标识：
      * `thread_idx` (tidx)：块内线程索引（0 到 bdim-1）
      * `block_idx` (bidx)：网格中的块索引
      * `block_dim` (bdim)：每个块的线程数
    - 全局线程ID计算为：`thread_idx = bidx * bdim + tidx`

2. **坐标映射**
    - 内核将每个线程的全局ID映射到2D张量坐标：
      * `ni = thread_idx % n`（列索引 - 更快变化的维度）
      * `mi = thread_idx // n`（行索引 - 更慢变化的维度）
    - 这种映射通过让相邻线程访问相邻内存位置来确保合并内存访问

3. **内存访问模式**
    - 每个线程：
      * 从张量A加载一个元素：`a_val = gA[mi, ni]`
      * 从张量B加载一个元素：`b_val = gB[mi, ni]`
      * 执行加法：`a_val + b_val`
      * 将结果存储到张量C：`gC[mi, ni] = result`
    - 内存考虑
      * 使用1:1线程到元素映射
      * 当warp中的线程访问连续元素时，内存访问是合并的
      * 没有显式使用共享内存或寄存器阻塞
      * 由于单个元素处理，隐藏内存延迟的能力有限

这个朴素实现为理解后续的更优化版本提供了基准，这些版本引入：
- 向量化内存访问
- 线程和值（TV）布局
- 高级平铺策略
- 自定义二进制操作

有关合并内存访问的更多详细信息，请阅读：https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/#coalesced-access-to-global-memory

### 内核启动配置和测试

本节演示如何：
1. 使用`cute.jit`函数配置和启动内核
2. 使用`torch`设置测试数据
3. 验证正确性

**启动配置**
  - 每个块使用256个线程（良好的占用率的常见选择）
  - 网格大小基于总元素计算：`(m * n) // threads_per_block`
  - 为简单起见，使用单维度块和网格配置

#### 主机JIT函数以启动内核

In [ ]:
@cute.jit  # Just-in-time compilation decorator
def naive_elementwise_add(
    mA: cute.Tensor,  # Input tensor A
    mB: cute.Tensor,  # Input tensor B
    mC: cute.Tensor,  # Output tensor C
):
    # Configure kernel launch parameters
    # --------------------------------
    # Choose number of threads per block
    # 256 is a common choice as it:
    # - Allows good occupancy on most GPUs
    # - Is a multiple of 32 (warp size)
    # - Provides enough threads for latency hiding
    num_threads_per_block = 256

    # Get input dimensions
    m, n = mA.shape  # Matrix dimensions (M rows × N columns)

    # Create kernel instance
    kernel = naive_elementwise_add_kernel(mA, mB, mC)

    # Launch kernel with calculated grid dimensions
    # -------------------------------------------
    # Grid size calculation:
    # - Total elements: m * n
    # - Blocks needed: ceil(total_elements / threads_per_block)
    # - Using integer division here assumes m * n is multiple of threads_per_block
    kernel.launch(
        grid=((m * n) // num_threads_per_block, 1, 1),  # Number of blocks in x,y,z
        block=(num_threads_per_block, 1, 1),  # Threads per block in x,y,z
    )

In [ ]:
# Test Setup
# ----------
# Define test dimensions
M, N = 16384, 8192  # Using large matrices to measure performance

# Create test data on GPU
# ----------------------
# Using float16 (half precision) for:
# - Reduced memory bandwidth requirements
# - Better performance on modern GPUs
a = torch.randn(M, N, device="cuda", dtype=torch.float16)  # Random input A
b = torch.randn(M, N, device="cuda", dtype=torch.float16)  # Random input B
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)  # Output buffer

# Calculate total elements for bandwidth calculations
num_elements = sum([a.numel(), b.numel(), c.numel()])

# Convert PyTorch tensors to CuTe tensors
# -------------------------------------
# from_dlpack creates CuTe tensor views of PyTorch tensors
# assumed_align=16 ensures proper memory alignment for vectorized access
a_ = from_dlpack(a, assumed_align=16)  # CuTe tensor A
b_ = from_dlpack(b, assumed_align=16)  # CuTe tensor B
c_ = from_dlpack(c, assumed_align=16)  # CuTe tensor C

In [ ]:
# Compile the kernel for the specific input types
naive_elementwise_add_ = cute.compile(naive_elementwise_add, a_, b_, c_)

# Run the kernel
naive_elementwise_add_(a_, b_, c_)

# Verify Results
# -------------
# Compare our kernel output with PyTorch's native implementation
torch.testing.assert_close(c, a + b)  # Raises error if results don't match

## Performance Analysis and Benchmarking

To understand and improve our kernel's performance, we need to measure its execution time and memory throughput. Let's analyze several key metrics:

* **Execution Time**
   - Measures raw kernel performance in microseconds
   - Lower is better
   - Affected by GPU clock speed, memory bandwidth, and kernel efficiency
* **Memory Throughput**
   - Measures how fast we can copy data (GB/s)
   - Higher is better
   - Theoretical peak varies by GPU model
   - For elementwise add:
     * Read: 2 elements (A and B)
     * Write: 1 element (C)
     * Total bytes = (2 reads + 1 write) × elements × sizeof(dtype)

Below is our benchmarking utility that measures these metrics:

In [ ]:
def benchmark(callable, a_, b_, c_):
    avg_time_us = cute.testing.benchmark(
        callable,
        kernel_arguments=cute.testing.JitArguments(a_, b_, c_),
        warmup_iterations=5,
        iterations=100,
    )

    # Calculate metrics
    # ----------------
    dtype = a_.element_type

    # Calculate total bytes transferred:
    # - 2 reads (A and B) + 1 write (C)
    # - Each element is dtype.width bits
    bytes_per_element = dtype.width // 8
    total_bytes = num_elements * bytes_per_element

    # Calculate achieved bandwidth
    achieved_bandwidth = total_bytes / (avg_time_us * 1000)  # GB/s

    # Print results
    # ------------
    print(f"Performance Metrics:")
    print(f"-------------------")
    print(f"Kernel execution time: {avg_time_us:.4f} us")
    print(f"Memory throughput: {achieved_bandwidth:.2f} GB/s")

In [ ]:
benchmark(naive_elementwise_add_, a_, b_, c_)
# Performance Metrics:
# -------------------
# Kernel execution time: 552.6870 us
# Memory throughput: 1457.07 GB/s

### Theoretical Analysis

This section analyze the performance characteristics and optimization opportunities of our elementwise addition kernel through several theoretical frameworks.

#### Little's Law

Little's Law provides crucial insights into relationship between latency, bandwidth and inflight operations:

$ L = \lambda \times W $

Where:
- $L$: Number of in-flight memory operations needed
- $\lambda$: Target memory bandwidth (bytes/cycle)
- $W$: Memory system latency (cycles)

According to *Little's Law*, naive implementation has
   - 1 element (4 bytes load + 2 bytes store) per thread
   - 256 threads/block × N blocks
   - Limited in-flight operations

In some GPUs, it's insufficient parallelism to saturate memory bandwidth.

#### Optimization Strategies

Based on this analysis, one commonly used technique is **Vectorization**. Instead of 1 element 
per load per thread, vectorization allows multiple element per load
   - Reduces instruction count
   - Improves memory coalescing
   - Increases operations in flight

In [ ]:
# 理论分析

本节通过多个理论框架分析我们的元素级加法内核的性能特征和优化机会。

# Little定律

Little定律提供了关于延迟、带宽和飞行中操作之间关系的重要见解：

# $ L = \lambda \times W $

# 其中：
# - $L$：在飞行中需要的内存操作数
# - $\lambda$：目标内存带宽（字节/周期）
# - $W$：内存系统延迟（周期）

# 根据*Little定律*，朴素实现具有
#    - 每个线程1个元素（4字节加载 + 2字节存储）
#    - 256个线程/块 × N个块
#    - 飞行中操作有限

# 在某些GPU上，并行性不足以饱和内存带宽。

# 优化策略

# 基于这个分析，一个常用的技术是**向量化**。不使用每次加载每个线程1个元素，
# 向量化允许每次加载多个元素
#    - 减少指令计数
#    - 改进内存合并
#    - 增加飞行中的操作

## Vectorized Load and Store

To improve performance according to Little's Law, we need to increase the number
of in-flight requests. We can do this by increasing the number of bytes handled
in each load & store operation per thread through vectorized memory access.

Since Ampere GPUs support up to 128-bit per load/store and each element is 16-bit,
we can load 8 elements per vectorized operation on contiguous rows.
CuTe tiling operations make this vectorization straightforward.

Using ``tiled_tensor = cute.zipped_divide(tensor, tiler)``, we can partition the input
``tensor`` into groups of ``tiler`` blocks. For vectorization, we specify ``tiler``
as the block of data each thread accesses (8 contiguous elements in the same row, or ``(1,8)``).
Different threads can then access different blocks by indexing into the 2nd mode of ``tiled_tensor``.

```python
mA : cute.Tensor                           # (2048,2048):(2048,1)
gA = cute.zipped_divide(a, tiler=(1, 8))   # tiled/vectorized => ((1,8),(2048,256)):((0,1),(2048,8))
```

$
    \begin{array}{ccccc}
    & ((1,8) & , & (2048,256)) & : ((0,1),(2048,8)) \\
    & \underbrace{\phantom{(1,8)}}_{tiler} & & \underbrace{\phantom{(2048,256)}}_{threads} & \\
    & \text{\scriptsize per-thread} & & \text{\scriptsize num of tiles}
    \end{array}
$

In [ ]:
## 向量化加载和存储

为了根据Little定律提高性能，我们需要增加飞行中请求的数量。我们可以通过向量化内存访问来实现这一点，即增加每个线程在每次加载和存储操作中处理的字节数。

由于Ampere GPU支持每次加载/存储最多128位，而每个元素是16位，我们可以在连续行上每次向量化操作加载8个元素。CuTe平铺操作使这种向量化变得简单。

使用``tiled_tensor = cute.zipped_divide(tensor, tiler)``，
我们可以将输入``tensor``分区为``tiler``块的组。
对于向量化，我们将``tiler``指定为每个线程访问的数据块（同一行中的8个连续元素，或``(1,8)``）。
不同的线程可以通过索引到``tiled_tensor``的第2模式来访问不同的块。


This vectorized kernel follows a similar structure to its naive non-vectorized counterpart,
with one key difference: the tensor slicing pattern. By using `(None, (mi, ni))` as the slice indices,
we can extract a `(1,8)` sub-tensor from `gA`, `gB` and `gC` like 

$ gA[(None, (mi, ni))]: $

$
  \begin{array}{ccccc}
    Layout: & ( & (1,8)                        & , & (2048,256) & )                    & : & ((0,1),(2048,8)) & \xrightarrow{\text{slice}} & ((1,8)):((0,1)) \\
            &   & \underbrace{\phantom{(1,8)}} &   & \underbrace{\phantom{(2048,256)}} &   & \\
    Coord:  & ( & None                         & , & (mi, ni)   & )                    &   &
  \end{array}
$

Then tensor data can be loaded into vector via the `gA[(None, (mi, ni))].load()` method. It is equivalent to

```python
v0 = gA[(0, (mi, ni))]   # => mA[(mi, ni * 8 + 0)]
v1 = gA[(1, (mi, ni))]   # => mA[(mi, ni * 8 + 1)]
v2 = gA[(2, (mi, ni))]   # => mA[(mi, ni * 8 + 2)]
v3 = gA[(3, (mi, ni))]   # => mA[(mi, ni * 8 + 3)]
v4 = gA[(4, (mi, ni))]   # => mA[(mi, ni * 8 + 4)]
v5 = gA[(5, (mi, ni))]   # => mA[(mi, ni * 8 + 5)]
v6 = gA[(6, (mi, ni))]   # => mA[(mi, ni * 8 + 6)]
v7 = gA[(7, (mi, ni))]   # => mA[(mi, ni * 8 + 7)]
```

### Assumed Alignment

In order to guide compile to use vectorized load/store, we must tell compiler to assume alignment of incoming pointer. 
It's on users side to guarantee actual pointer at runtime meet the alignment restriction.

```python
a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

# Compile kernel with alignment assumption
compiled_func = cute.compile(vectorized_elementwise_add, a_, b_, c_)
```

It's worth to note that partitioned or tiled tensor could have different alignment of its base pointer because of offset
during sub-slice.

In [ ]:
@cute.jit
def vectorized_elementwise_add(mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
    threads_per_block = 256

    gA = cute.zipped_divide(mA, (1, 8))
    gB = cute.zipped_divide(mB, (1, 8))
    gC = cute.zipped_divide(mC, (1, 8))

    print("[DSL INFO] Tiled Tensors:")
    print(f"[DSL INFO]   gA = {gA}")
    print(f"[DSL INFO]   gB = {gB}")
    print(f"[DSL INFO]   gC = {gC}")

    vectorized_elementwise_add_kernel(gA, gB, gC).launch(
        grid=(cute.size(gC, mode=[1]) // threads_per_block, 1, 1),
        block=(threads_per_block, 1, 1),
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)

a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

compiled_func = cute.compile(vectorized_elementwise_add, a_, b_, c_)
compiled_func(a_, b_, c_)

# verify correctness
torch.testing.assert_close(c, a + b)

In [ ]:
benchmark(compiled_func, a_, b_, c_)
# Performance Metrics:
# -------------------
# Kernel execution time: 541.7709 us
# Memory throughput: 1486.43 GB/s


## TV Layout

Both the naive and vectorized kernels follow a common pattern to map thread indices
to physical addresses in two steps:

Step 1: Map thread index to logical coordinates in `(M, N)`

* `mi = thread_idx // n`
* `ni = thread_idx % n`

In native version, each thread process 1 element, in this case, `mi` and `ni` is logical
coordinate into data tensor `mA`, `mB` and `mC`.

Int vectorized version, each thread process multiple values of input and output tensor.
logical coordinate should be computed with both thread and value index.

* `thread_idx // n`
* `(thread_idx % n) * 8 + value_idx`


Step 2: Map logical coordinates in `(M, N)` to physical addresses using the tensor layout

* Vectorized Load

```python
    frgA = gA[(None, (mi, ni))].load()
```

* Elementwise Load (less efficient)

```python
    frgA0 = mA[(mi, ni * 8 + 0)]
    frgA1 = mA[(mi, ni * 8 + 1)]
    frgA2 = mA[(mi, ni * 8 + 2)]
    frgA3 = mA[(mi, ni * 8 + 3)]
    frgA4 = mA[(mi, ni * 8 + 4)]
    frgA5 = mA[(mi, ni * 8 + 5)]
    frgA6 = mA[(mi, ni * 8 + 6)]
    frgA7 = mA[(mi, ni * 8 + 7)]

    # Or use divided layout

    frgA0 = gA[(0, (mi, ni))]
    frgA1 = gA[(1, (mi, ni))]
    frgA2 = gA[(2, (mi, ni))]
    frgA3 = gA[(3, (mi, ni))]
```

CuTe introduces TV layout to represent this mapping from thread index and value index
(i.e., the 4 elements loaded per thread) to the logical coordinate space of a tensor.
By configuring different TV layouts, we can experiment with different memory access
patterns with minimal code changes.

**Definition:** *TV Layout* is rank-2 layout which maps `(thread_index, value_index)` 
to logical coordinate of tensor.  

We always have *TV Layout* with canonical form as `(thread_domain, value_domain):(..., ...)`.

With *TV Layout*, each thread can find logical coordinates or indices of data partitioned
to current thread.


### 深度解释：什么是 TV Layout？

在 CUDA 编程中，
最头疼的问题通常是：“我是第 $t$ 号线程，我该读内存里的哪几个数？”

传统的做法是写一堆复杂的索引抵消公式（如 idx = threadIdx.x * 8 + i）。

CuTe 通过 TV Layout 把这个映射关系对象化了。

(1) 核心公式：坐标的映射TV Layout 本质上是一个函数：$$f(thread\_idx, value\_idx) \rightarrow (logical\_coord\_M, logical\_coord\_N)$$

Thread Domain：参与计算的所有线程集合（例如一个 Block 里的 256 个线程）。

Value Domain：每个线程负责处理的数据量（例如向量化宽度为 8）。

(2) 为什么叫 "TV"？
    T (Thread)：代表硬件线程。
    V (Value)：代表该线程持有的数据（通常在寄存器中）。

当你定义了一个 TV Layout，你就定义了数据是如何“分发”给线程的。

例如：连续分布：线程 0 处理元素 0-7，线程 1 处理元素 8-15。
交错分布（Strided）：线程 0 处理元素 0, 256, 512...（这在避免共享内存 Bank Conflict 时非常有用）。

(3) 在代码中的威力看这个例子：PythonfrgA = gA[(None, (mi, ni))].load()

这里的 (None, (mi, ni)) 实际上是在利用分块后的布局。
当你使用 gA = zipped_divide(mA, TV_Layout) 时，CuTe 自动帮你把原始张量 mA 按照线程和值的逻辑重新排布了。

None：在 CuTe 索引中通常代表“取这个维度的全部”。在这里意味着“取当前线程对应的所有 Value”。这就实现了一次性从显存中执行 向量化加载（Vectorized Load），将 8 个 float16 直接搬进寄存器。

## Elementwise with TV Layout

In this example, we rewrite elementwise kernel with two levels of tiling: 
* the thread-block level 
* the thread level with TV Layout and tiling

For thread-block level tiling, each input & output tensor is first divided
into a group of ``(TileM, TileN)`` sub-tensors at the host side. Please be noticed that
in this case, we still use `zipped_divide` but for tiling at thread-block level.

Inside the GPU kernel, we slice tiled tensor with the thread-block index at the 2nd mode 
as ``gA[((None, None), bidx)]``, which returns a thread-block local view of
a single ``(TileM, TileN)`` sub-tensor. This sub-tensor maps logical coordinates
inside ``(TileM, TileN)`` to physical address of elements.

At thread level tiling, we compose the above sub-tensor (logical coordinates to physical addresses) 
with the TV layout (thread & value indices to logical coordinates). This gives us a tiled sub-tensor 
that maps from thread & value indices directly to physical addresses.

We then slice it with the thread index as ``tidfrgA[(tidx, None)]`` to get 
a thread-local view of the data each thread accesses. Note that the thread index
is now in the 1st mode, as TV layout is normally have form ``(thread_domain, value_domain):(...)``.

### Kernel Code

### Host Code

The host code below shows the construction of the TV layout. By composing
a thread layout of ``(4,64):(64,1)`` (64 threads read contiguous elements on the row dimension,
then 64-thread-groups(2 warps) read different rows) with a value layout of ``(16,8):(8,1)`` (each thread reads
8 contiguous 16b elements on the row dimension across 4 contiguous rows).

In order to generalize, we started with byte-layout to describe layout for elements in bytes. This is
to ensure use of 128bit vectorized load store. Then we leverage ``recast_layout`` to convert into
element-layout.

```python
    # src type bits: 8
    # dst type bits: bits of element type
    val_layout = cute.recast_layout(dtype.width, 8, bit_val_layout)
```

In [ ]:

@cute.jit
def elementwise_add(
    mA: cute.Tensor,
    mB: cute.Tensor,
    mC: cute.Tensor,
):
    # mA layout: (M, N):(N, 1)
    # TV layout map thread & value index to (64, 512) logical tile
    #  - contiguous thread index maps to mode-1 because input layout is contiguous on
    #     mode-1 for coalesced load-store
    #  - each thread load contiguous 16 bytes each row and load 16 rows
    coalesced_ldst_bytes = 16

    # Compile time validation: expect same element type for all input tensors
    assert all(t.element_type == mA.element_type for t in [mA, mB, mC])
    dtype = mA.element_type
    # 创建 thread layout 4 rows, 64 threads pre row(== 每行两个warp)
    thr_layout = cute.make_ordered_layout((4, 64), order=(1, 0))
    # 创建 val layout 16 rows, 16 bytes(128bit) pre / row
    val_layout = cute.make_ordered_layout((16, coalesced_ldst_bytes), order=(1, 0))
    print('val_layout1: {}'.format(val_layout))
    # recast_layout 将 val layout 转化为element 粒度的layout
    val_layout = cute.recast_layout(dtype.width, 8, val_layout)
    print('val_layout2: {}'.format(val_layout))
    # make_layout_tv 返回 tile 大小 tv_layout value 到 thread 的映射
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)
#   cutlass.utils.print_latex_tv(tv_layout, tiler_mn)
    print(f"[DSL INFO] Tiler: {tiler_mn}")
    print(f"[DSL INFO] TV Layout: {tv_layout}")

    gA = cute.zipped_divide(mA, tiler_mn)  # ((TileM, TileN), (RestM, RestN))
    gB = cute.zipped_divide(mB, tiler_mn)  # ((TileM, TileN), (RestM, RestN))
    gC = cute.zipped_divide(mC, tiler_mn)  # ((TileM, TileN), (RestM, RestN))

    print("Tiled Input Tensors:")
    print("[DSL INFO] Tiled Tensors:")
    print(f"[DSL INFO]   gA = {gA.type}")
    print(f"[DSL INFO]   gB = {gB.type}")
    print(f"[DSL INFO]   gC = {gC.type}")
    print(f"cute.size(gC, mode=[1]) {cute.size(gC, mode=[1])}")

    # Launch the kernel asynchronously
    # Async token(s) can also be specified as dependencies
    elementwise_add_kernel(gA, gB, gC, tv_layout).launch(
        grid=[cute.size(gC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)

a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

elementwise_add_ = cute.compile(elementwise_add, a_, b_, c_)
elementwise_add_(a_, b_, c_)

# verify correctness
torch.testing.assert_close(c, a + b)

# thr_layout: (4,64):(64,1)
# 
# val_layout1: (16,16):(16,1)
# dtype.width: 16
# 每个thread负责的数据区域
# val_layout2: (16,8):(8,1)
# make_layout_tv后 64 = 16 * 4，512 = 64 * 8
# [DSL INFO] Tiler: (64, 512)
# 线程维度 64个thread * 4 rows， 值维度：8个val * 16 行
#  stride 
# [DSL INFO] TV Layout: ((64,4),(8,16)):((512,16),(64,1))
# Tiled Input Tensors:
# [DSL INFO] Tiled Tensors:
# [DSL INFO]   gA = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
# [DSL INFO]   gB = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
# [DSL INFO]   gC = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
# cute.size(gC, mode=[1]) 4096
# Composed with TV layout:
#   tidfrgA: !cute.memref<f16, gmem, align<16>, "((64,4),(8,16)):((8,131072),(1,8192))">

### 1. 假设我们的设定假设我们正在处理一个巨大的矩阵，我们想定义一个 Thread Block 如何处理其中的一小块数据（Tile）。

* 线程布局 (thr_layout): 
(2, 4)表示一个线程块里有 8 个线程。逻辑上排成 2 行 4 列。

* 值布局 (val_layout):
(4, 2)表示每个线程负责处理 8 个元素。这 8 个元素在它手里排成 4 行 2 列。

2. make_layout_tv 产生的第一个产物：
#### tiler_mn (模具的大小)
tiler_mn 定义了这个模具在原始矩阵上覆盖的总面积。

它的计算非常直观，就是**“线程数 $\times$ 每个线程持有的数据量”**。

行方向 (M): $2 \text{ (线程行)} \times 4 \text{ (值行)} = 8$

列方向 (N): $4 \text{ (线程列)} \times 2 \text{ (值列)} = 8$
所以，tiler_mn 是一个 $8 \times 8$ 的形状。

当你执行 gA = zipped_divide(mA, tiler_mn) 时，
CuTe 就会把原始大矩阵切成无数个 $8 \times 8$ 的小方块。

3. 第二个产物：tv_layout (内部的导航地图)

这是最容易搞混的地方。
tv_layout 的作用是回答：“在 $8 \times 8$ 的方块里，第 $T$ 号线程拿到的第 $V$ 个值，对应方块里坐标 $(m, n)$ 的哪一个？”它建立了一个映射关系：$(Thread, Value) \rightarrow (M, N)$。

举例说明映射过程：如果 thr_layout 和 val_layout 的 order 都是默认的（先变行，再变列）：
对于 0 号线程 (T0)：
它的 Value 0 会映射到 Tile 的 $(0, 0)$
它的 Value 1 会映射到 Tile 的 $(1, 0)$ ... (因为它有 4 行数据)

对于 1 号线程 (T1)：
由于它是 thr_layout 的第一行、第二个线程（列变了），
它的数据会从 Tile 的列坐标 $1$ 开始。
它的 Value 0 映射到 Tile 的 $(0, 1)$。

4. 为什么这个函数如此重要？
如果你不使用 make_layout_tv，
在写 CUDA Kernel 时，你必须手动写这种代码：
C++int m = (tid / 4) * 4 + (vid / 2);
int n = (tid % 4) * 2 + (vid % 2);
float val = A[m * N + n];

这种代码不仅难读，而且一旦你想把 FP16 改成 FP8，或者想把 256 线程改成 128 线程，
你得重写所有的算式。而在 CuTe 中：
你修改 thr_layout 或 val_layout 的定义。
make_layout_tv 自动生成新的 tv_layout。
内核里的 gA[tid, vid] 会自动指向正确的内存地址。

5. 总结

tiler_mn: 是宏观的。它告诉 zipped_divide 怎么把大矩阵切成小块（Tile）。

tv_layout: 是微观的。它告诉内核里的每一个线程，在小块（Tile）内部如何定位自己的数据。